In [1]:

import numpy as np
from time import time
import pickle
from scipy.optimize import minimize, OptimizeResult

from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import QAOAAnsatz
from qiskit.transpiler.passes.routing.commuting_2q_gate_routing import SwapStrategy

from qiskit_aer import AerSimulator
from qiskit_aer.primitives import SamplerV2 as Sampler, EstimatorV2 as Estimator

from qiskit_ibm_runtime.fake_provider import FakeFez

from qopt_best_practices.sat_mapping import SATMapper

from qiskit_qaoa.utils.circuit_graph_utils import circuit_to_graph, graph_to_operator, circuit_construction
from qiskit_qaoa.utils.hamiltonian_utils import get_Q_and_hamiltonian
from qiskit_qaoa.utils.string_utils import evaluate_sparse_pauli_samples



filename = 'trivial'
p: int = 4
hardware = True
shots = 1024
noisy = False
init_type = 'ramp'

seed = 1
rng = np.random.default_rng(seed=seed)

backend_options = dict(
    method='statevector',
    device='CPU',
    precision='single'
)
fake_fez = FakeFez()
backend = AerSimulator.from_backend(fake_fez, **backend_options)

data_file = f'/lustre/scratch127/qpg/jc59/out/oriented/qubo_data_{filename}.gfa.pkl'

Q, hamiltonian, offset = get_Q_and_hamiltonian(data_file)
qc = QAOAAnsatz(
    cost_operator=hamiltonian,
    reps = p,
    flatten=True
)
transpiled_qc = transpile(qc, backend, optimization_level=3, seed_transpiler=seed)


def print_circuit_info(qc, circuit_name):
    print(f'{circuit_name} has {qc.count_ops().get("cz", 0) + qc.count_ops().get("rzz", 0) + qc.count_ops().get("cx", 0)} 2Q gates \
    and {qc.depth(lambda instr: len(instr.qubits) > 1)} 2Q depth')


print_circuit_info(transpiled_qc, '(Transpiled) Circuit')

graph = circuit_to_graph(qc, qc.parameters[p])

swap_strat = SwapStrategy.from_line(range(graph.order()))
edge_coloring = {(idx, idx + 1): (idx + 1) % 2 for idx in range(graph.order())}

remapped_g, sat_map, min_sat_layers = SATMapper(timeout=30).remap_graph_with_sat(
    graph=graph, swap_strategy=swap_strat
)
if remapped_g is None:
    raise Exception('Failed to find initial layout')

cost_op = graph_to_operator(remapped_g)
singles = cost_op[cost_op.paulis.z.sum(axis=-1) == 1]
doubles = cost_op[cost_op.paulis.z.sum(axis=-1) == 2]

circ_dict = circuit_construction(singles, doubles, backend, swap_strat, edge_coloring, {}, p)

backend_circ = circ_dict["backend"]
print_circuit_info(backend_circ, '(Transpiled) Remapped Circuit')


if hardware:
    # transpiled again for the FakeFez backend
    circuit: QuantumCircuit = circ_dict["backend"]
else:
    backend = AerSimulator(**backend_options)
    circuit: QuantumCircuit = circ_dict["circuit_to_sample"]

qaoa_depth = len(circuit.parameters) // 2


if init_type == 'ramp':
    t = 0.7 * p
    betas = np.linspace(
        (1 / p) * (t * (1 - 0.5 / p)), (1 / p) * (t * 0.5 / p), p
    )
    gammas = betas[::-1]
    init_params = betas.tolist() + gammas.tolist()
else:
    init_params = rng.uniform(0, 0.9 * np.pi, qaoa_depth).tolist() + rng.uniform(0, 0.5 * np.pi, qaoa_depth).tolist()
print(f'Init: {init_params}')

if noisy:
    sampler = Sampler.from_backend(backend=backend, seed=seed)
else:
    sampler = Sampler(seed=seed, options=dict(backend_options=backend_options))
print(f'Noise model: {getattr(sampler._backend.options, "noise_model", "Ideal noise")}')

history = []


def callback(intermediate_result: OptimizeResult):
    print(f'Current params: {intermediate_result.x}. Current func value: {intermediate_result.fun}')
    if intermediate_result.fun == -1:
        raise StopIteration
    

def callback_cobyla(xk: np.ndarray):
    print(f'Current params: {xk}.')


def cvar(energies, alpha=1.0):
    sorted_energies = sorted(energies)
    end_idx = int(alpha * len(energies))
    return np.sum(sorted_energies[0:end_idx]) / end_idx


def objective(x: np.ndarray):
    start = time()
    assigned_circuit = circuit.assign_parameters(x, inplace=False)
    sampler_job = sampler.run([assigned_circuit], shots=shots)
    sampler_result = sampler_job.result()
    counts = sampler_result[0].data.c.get_counts()
    sampling_time = time() - start
    start = time()
    energies = []
    # evals = evaluate_sparse_pauli_samples(counts.keys(), cost_op) + offset
    int_samples = [np.array([int(x) for x in sample[::-1]]) for sample in counts.keys()]
    evals = [
        sample @ Q @ sample for sample in int_samples
    ]
    energies = [count * [evals[idx]] for idx, count in enumerate(counts.values())]
    flat_energies = [x for xs in energies for x in xs]
    total_energy = cvar(flat_energies, 0.25)

    classical_post_process_time = time() - start
    history.append((sampling_time, total_energy, x.tolist(), counts, classical_post_process_time))
    return total_energy

(Transpiled) Circuit has 3780 2Q gates     and 1221 2Q depth
(Transpiled) Remapped Circuit has 2268 2Q gates     and 232 2Q depth
Init: [0.6124999999999999, 0.4375, 0.2625, 0.0875, 0.0875, 0.2625, 0.4375, 0.6124999999999999]
Noise model: None


In [2]:
# evaluate_sparse_pauli_samples(
#     [
#         '0010000'+'0000100'+'0000001',
#         '1000000'+'0000100'+'0000001',
#         '0100000'+'0000100'+'0000001',
#         '0000100'+'0000100'+'0000001',
#         '0000100'+'0000100'+'1100011'
#     ], 
#     hamiltonian
# ) + offset

In [3]:
(cost_op.sort(weight=True) -hamiltonian.apply_layout(list(sat_map.values()), hamiltonian.num_qubits).sort(weight=True)).simplify()

SparsePauliOp(['IIIIIIIIIIIIIIIIIIIII'],
              coeffs=[0.+0.j])

In [4]:
# inv_map = {v: k for k, v in sat_map.items()}
# sample = '0010000'+'0000100'+'0000001'
# new_sample = [''] * len(sample)
# for x in range(len(sample)):
#     new_sample[len(sample)-1- sat_map[len(sample)-1-x]] = sample[x]

# evaluate_sparse_pauli_samples(
#     [
#         ''.join(new_sample),
#         '101011100001100011111'
#     ], 
#     cost_op
# ) + offset

In [5]:
# objective(np.array([0]*8))

In [6]:
x = np.array([0]*8)
assigned_circuit = circuit.assign_parameters(x, inplace=False)
sampler_job = sampler.run([assigned_circuit], shots=shots)
sampler_result = sampler_job.result()

In [7]:
counts = sampler_result[0].data.c.get_counts()
int_samples = [np.array([int(x) for x in sample[::-1]]) for sample in counts.keys()]
evals = np.array([
    sample @ Q @ sample for sample in int_samples
]) + offset
energies = [count * [evals[idx]] for idx, count in enumerate(counts.values())]
flat_energies = [x for xs in energies for x in xs]
total_energy = cvar(flat_energies, 0.25)

In [8]:
np.argmin(evals)

np.int64(810)

In [9]:
int_samples[810]

array([1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0])

In [10]:
sol = [1,0,0,0,0,0,0] +  [0,0,1,0,0,0,0] +  [0,0,0,0,1,0,0]
sol @ Q @ sol + offset

np.float64(0.0)

In [11]:
sol = [1,0,0,0,0,0,0] +  [0,0,0,0,1,0,0] +  [0,0,0,0,0,0,1]
sol @ Q @ sol + offset

np.float64(0.1875)

In [12]:
# method = "COBYLA"
# result = minimize(
#     objective, x0=init_params, 
#     method=method, 
#     bounds=tuple((0,1) for _ in range(2 * p)), 
#     options={"maxiter": 100, "maxfev": 100},  # "rhobeg": 0.01, "ftol": 1e-7
#     callback=callback if method not in ['SLSQP', 'COBYLA', 'TNC'] else callback_cobyla
# )
# print(result)

In [1]:
from scipy.optimize import minimize, show_options


In [2]:
show_options('minimize', 'COBYLA')

Minimize a scalar function of one or more variables using the
Constrained Optimization BY Linear Approximation (COBYLA) algorithm.

Options
-------
rhobeg : float
    Reasonable initial changes to the variables.
tol : float
    Final accuracy in the optimization (not precisely guaranteed).
    This is a lower bound on the size of the trust region.
disp : bool
    Set to True to print convergence messages. If False,
    `verbosity` is ignored as set to 0.
maxiter : int
    Maximum number of function evaluations.
catol : float
    Tolerance (absolute) for constraint violations
